# Solution to p22

After score thresholding, rows 0, 1, and 2 remain; row 3 is removed because `.28 < .30`, even though its truth IoU is `.60`. It is a thresholded candidate, not a missed object: the only stated truth box is already matched by retained row 0 (IoU `.76`), so dropping row 3 costs no recall here. It would become a false negative only if it were the sole match for a separate truth object; the diagnostic for that case is precision-recall/score-calibration curves on validation data, and then lowering or calibrating the score threshold if the predeclared operating point supports it.

NMS selects row 0 first (`.92`). Row 1 is the duplicate: it is the same class and has IoU `.81 > .50` to row 0, so it is suppressed. If duplicates are frequent before NMS, diagnose assignment/matching and duplicate-positive training; the exact inference repair is to retain same-class NMS with suppression for IoU strictly above `.50` (and tune that threshold only on validation data).

Row 2 survives: score `.34` passes threshold and IoU `.02` to every other prediction prevents NMS suppression, but truth IoU zero makes it a retained false positive. Diagnose objectness calibration and hard-negative errors; an exact pipeline intervention is to calibrate/raise the score threshold using validation precision-recall evidence or train with additional train-only hard negatives.

Class-aware NMS partitions by predicted class. Therefore even a highly overlapping different-class box is not compared for suppression with row 0; it remains for class-specific evaluation or a later task-specific rule.

### Answer check

In [ ]:
rows={0:"retained true positive",1:"duplicate suppressed",2:"retained false positive",3:"thresholded candidate"}
assert .28 < .30 <= .34
assert .81 > .50
assert rows[1]=="duplicate suppressed"
assert rows[2]=="retained false positive"
assert rows[3]=="thresholded candidate"
assert .76 >= .50  # row 0 already matches the truth, so row 3 is not a missed object
